# CLIP Zero-Shot Baseline — hymenium_type

Establish a free performance floor for morphological feature classification
using CLIP zero-shot inference. No training, no labels — just text prompts
matched against image embeddings.

**Model:** `openai/clip-vit-base-patch16` via HuggingFace transformers

**Target feature:** `hymenium_type` (7 classes: gills, ridges, pores, teeth, smooth, gleba, alveolate)

**Expected output:** Per-class accuracy table showing which features CLIP
can already distinguish well (>80%) vs. which need supervised DINOv2 heads (<60%).

In [ ]:
import sys
from pathlib import Path

import torch
import yaml
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

PROJECT_ROOT = Path.cwd().parents[1]  # vision/notebooks/ -> project root
sys.path.insert(0, str(PROJECT_ROOT))

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Load CLIP model

In [ ]:
MODEL_NAME = "openai/clip-vit-base-patch16"

model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
print(f"Loaded {MODEL_NAME}")

## 2. Define text prompts for hymenium_type

Descriptive prompts grounded in morphological vocabulary definitions.
Each prompt describes the visual appearance of the hymenium type.

In [ ]:
# Load feature config for class list
with open(PROJECT_ROOT / "vision" / "config" / "features.yaml") as f:
    features_config = yaml.safe_load(f)

HYMENIUM_CLASSES = features_config["hymenium_type"]["classes"]
print(f"hymenium_type classes ({len(HYMENIUM_CLASSES)}): {HYMENIUM_CLASSES}")

# Descriptive prompts — one per class, grounded in vocabulary definitions
HYMENIUM_PROMPTS = {
    "gills": "a mushroom with blade-like gills under the cap",
    "ridges": "a mushroom with blunt forking ridges under the cap",
    "pores": "a mushroom with a sponge-like pore surface under the cap",
    "teeth": "a mushroom with hanging spines or teeth under the cap",
    "smooth": "a mushroom with a smooth surface under the cap",
    "gleba": "a spherical or oval shaped mushroom with enclosed internal spore mass",
    "alveolate": "a mushroom with deeply pitted honeycomb surface over the cap",
}

# Verify all classes have prompts
assert set(HYMENIUM_PROMPTS.keys()) == set(HYMENIUM_CLASSES)
prompt_texts = [HYMENIUM_PROMPTS[c] for c in HYMENIUM_CLASSES]
print("\nPrompts:")
for cls, prompt in zip(HYMENIUM_CLASSES, prompt_texts):
    print(f"  {cls:12s} → {prompt}")

## 3. Pre-encode text prompts

Encode all text prompts once. These are reused for every image.

In [ ]:
with torch.no_grad():
    text_inputs = processor(text=prompt_texts, return_tensors="pt", padding=True).to(DEVICE)
    text_out = model.get_text_features(**text_inputs)
    # transformers 5.x returns BaseModelOutputWithPooling; pooler_output is already projected
    text_embeds = text_out.pooler_output if hasattr(text_out, "pooler_output") else text_out
    text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

print(f"Text embeddings shape: {text_embeds.shape}")  # (7, 512)

## 4. Load test images

Point `IMAGE_DIR` to a directory of test images. Expected structure:
```
test_images/
├── gills/
│   ├── img1.jpg
│   └── img2.jpg
├── pores/
│   ├── img3.jpg
│   └── ...
└── ...
```

Each subfolder name is the ground-truth hymenium_type label.
Target: ~5-10 images per class, ~35-70 images total.

In [ ]:
# ── Configure image directory ──
# Change this path to point to your test images organized by class
IMAGE_DIR = PROJECT_ROOT / "data" / "images" / "test_clip" / "hymenium_type"

# Scan for images
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
test_samples = []  # list of (path, ground_truth_label)

if IMAGE_DIR.exists():
    for class_dir in sorted(IMAGE_DIR.iterdir()):
        if not class_dir.is_dir():
            continue
        label = class_dir.name
        if label not in HYMENIUM_CLASSES:
            print(f"  WARNING: skipping unknown class dir '{label}'")
            continue
        for img_path in sorted(class_dir.iterdir()):
            if img_path.suffix.lower() in IMAGE_EXTENSIONS:
                test_samples.append((img_path, label))

    print(f"Found {len(test_samples)} test images across classes:")
    from collections import Counter
    dist = Counter(label for _, label in test_samples)
    for cls in HYMENIUM_CLASSES:
        print(f"  {cls:12s}: {dist.get(cls, 0)}")
else:
    print(f"Image directory not found: {IMAGE_DIR}")
    print("Create it with subdirectories per hymenium_type class, each containing test images.")
    print("Then re-run this cell.")

## 5. Zero-shot classification

For each test image:
1. Encode with CLIP image encoder
2. Compute cosine similarity against all text prompts
3. Apply softmax to get probabilities
4. Record prediction and ground truth

In [ ]:
def classify_image(image_path: Path) -> dict:
    """Zero-shot classify a single image against hymenium_type prompts.

    Returns dict with predicted class, confidence, and full probability distribution.
    """
    image = Image.open(image_path).convert("RGB")

    with torch.no_grad():
        image_inputs = processor(images=image, return_tensors="pt").to(DEVICE)
        image_out = model.get_image_features(**image_inputs)
        # transformers 5.x returns BaseModelOutputWithPooling
        image_embeds = image_out.pooler_output if hasattr(image_out, "pooler_output") else image_out
        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)

        # Cosine similarity (image vs all text prompts), scaled by CLIP temperature
        logit_scale = model.logit_scale.exp()
        logits = (image_embeds @ text_embeds.T) * logit_scale
        probs = logits.softmax(dim=-1).squeeze(0).cpu()

    pred_idx = probs.argmax().item()
    return {
        "predicted": HYMENIUM_CLASSES[pred_idx],
        "confidence": probs[pred_idx].item(),
        "probs": {cls: probs[i].item() for i, cls in enumerate(HYMENIUM_CLASSES)},
    }


# Run classification on all test images
assert len(test_samples) > 0, "No test images found — populate IMAGE_DIR first"

results = []
for img_path, gt_label in test_samples:
    result = classify_image(img_path)
    result["ground_truth"] = gt_label
    result["correct"] = result["predicted"] == gt_label
    result["image"] = img_path.name
    results.append(result)

correct = sum(r["correct"] for r in results)
print(f"Overall accuracy: {correct}/{len(results)} ({100 * correct / len(results):.1f}%)")

## 6. Results — per-class accuracy and confusion matrix

In [ ]:
import pandas as pd
from collections import Counter

# Per-class accuracy table
rows = []
for cls in HYMENIUM_CLASSES:
    cls_results = [r for r in results if r["ground_truth"] == cls]
    n = len(cls_results)
    if n == 0:
        rows.append({"class": cls, "n": 0, "correct": 0, "accuracy": None})
        continue
    n_correct = sum(r["correct"] for r in cls_results)
    rows.append({"class": cls, "n": n, "correct": n_correct, "accuracy": n_correct / n})

df_acc = pd.DataFrame(rows)
df_acc["accuracy_pct"] = df_acc["accuracy"].apply(lambda x: f"{100*x:.1f}%" if x is not None else "N/A")
print("Per-class accuracy:")
print(df_acc[["class", "n", "correct", "accuracy_pct"]].to_string(index=False))
print()

# Confusion matrix
gt_labels = [r["ground_truth"] for r in results]
pred_labels = [r["predicted"] for r in results]

confusion = pd.DataFrame(0, index=HYMENIUM_CLASSES, columns=HYMENIUM_CLASSES)
for gt, pred in zip(gt_labels, pred_labels):
    confusion.loc[gt, pred] += 1

print("Confusion matrix (rows=ground truth, cols=predicted):")
print(confusion)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: per-class accuracy
ax = axes[0]
accs = [r["accuracy"] if r["accuracy"] is not None else 0 for r in rows]
colors = ["#2ecc71" if a >= 0.8 else "#f39c12" if a >= 0.6 else "#e74c3c" for a in accs]
ax.barh(HYMENIUM_CLASSES, accs, color=colors)
ax.set_xlim(0, 1)
ax.set_xlabel("Accuracy")
ax.set_title("CLIP Zero-Shot Accuracy by Class")
ax.axvline(0.8, color="green", linestyle="--", alpha=0.5, label="80% (good)")
ax.axvline(0.6, color="orange", linestyle="--", alpha=0.5, label="60% (needs DINOv2)")
ax.legend(loc="lower right", fontsize=8)

# Confusion matrix heatmap
ax = axes[1]
conf_arr = confusion.values.astype(float)
# Normalize by row (ground truth)
row_sums = conf_arr.sum(axis=1, keepdims=True)
conf_norm = np.divide(conf_arr, row_sums, where=row_sums > 0, out=np.zeros_like(conf_arr))

im = ax.imshow(conf_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(HYMENIUM_CLASSES)))
ax.set_yticks(range(len(HYMENIUM_CLASSES)))
ax.set_xticklabels(HYMENIUM_CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(HYMENIUM_CLASSES, fontsize=8)
ax.set_xlabel("Predicted")
ax.set_ylabel("Ground Truth")
ax.set_title("Normalized Confusion Matrix")

# Annotate cells
for i in range(len(HYMENIUM_CLASSES)):
    for j in range(len(HYMENIUM_CLASSES)):
        val = conf_arr[i, j]
        if val > 0:
            ax.text(j, i, f"{int(val)}", ha="center", va="center",
                    color="white" if conf_norm[i, j] > 0.5 else "black", fontsize=8)

fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 7. Detailed misclassifications

Show which images were misclassified and what they were confused with.

In [ ]:
errors = [r for r in results if not r["correct"]]
if errors:
    print(f"{len(errors)} misclassifications:\n")
    for r in errors:
        print(f"  {r['image']:30s}  GT={r['ground_truth']:12s}  Pred={r['predicted']:12s}  Conf={r['confidence']:.2f}")
        # Show top-2 probabilities
        sorted_probs = sorted(r["probs"].items(), key=lambda x: -x[1])[:3]
        top_str = ", ".join(f"{k}={v:.2f}" for k, v in sorted_probs)
        print(f"    top-3: {top_str}")
else:
    print("No misclassifications — perfect zero-shot accuracy!")

## 8. Summary

| Accuracy Range | Interpretation | Action |
|---|---|---|
| >80% | CLIP handles this well | May not need supervised head |
| 60-80% | Moderate | DINOv2 head likely helps |
| <60% | Poor zero-shot | Priority target for DINOv2 head |

In [ ]:
print("=== CLIP Zero-Shot Baseline Summary for hymenium_type ===\n")
for r in rows:
    cls = r["class"]
    acc = r["accuracy"]
    if acc is None:
        tag = "NO DATA"
    elif acc >= 0.8:
        tag = "GOOD (may skip DINOv2 head)"
    elif acc >= 0.6:
        tag = "MODERATE (DINOv2 head likely helps)"
    else:
        tag = "POOR (priority DINOv2 target)"
    acc_str = f"{100*acc:.0f}%" if acc is not None else "N/A"
    print(f"  {cls:12s}  {acc_str:>5s}  {tag}")

print(f"\nOverall: {correct}/{len(results)} ({100*correct/len(results):.1f}%)")

## 9. Comparison: DB habitus vs iNaturalist detail images

The DB source images are mostly top/side habitus shots where the hymenium
is not visible. To test whether CLIP improves when the feature IS visible,
we run the same classifier on a second test set of iNaturalist photos
chosen for underside/detail views and taxa with prominent hymenium features.

This separates two failure modes:
1. **View angle** — image doesn't show the hymenium at all
2. **Model capability** — CLIP can't distinguish the feature even when visible

In [ ]:
# ── Load iNaturalist detail/underside test set ──
INAT_DIR = PROJECT_ROOT / "data" / "images" / "test_clip_underside" / "hymenium_type"

inat_samples = []
if INAT_DIR.exists():
    for class_dir in sorted(INAT_DIR.iterdir()):
        if not class_dir.is_dir():
            continue
        label = class_dir.name
        if label not in HYMENIUM_CLASSES:
            continue
        for img_path in sorted(class_dir.iterdir()):
            if img_path.suffix.lower() in IMAGE_EXTENSIONS:
                inat_samples.append((img_path, label))

    inat_dist = Counter(label for _, label in inat_samples)
    print(f"iNat detail set: {len(inat_samples)} images")
    for cls in HYMENIUM_CLASSES:
        print(f"  {cls:12s}: {inat_dist.get(cls, 0)}")
else:
    print(f"iNat image directory not found: {INAT_DIR}")
    print("Run: python -m vision.scripts.fetch_inat_test_images")

In [ ]:
# ── Classify iNat images ──
assert len(inat_samples) > 0, "No iNat test images — run fetch_inat_test_images.py first"

inat_results = []
for img_path, gt_label in inat_samples:
    result = classify_image(img_path)
    result["ground_truth"] = gt_label
    result["correct"] = result["predicted"] == gt_label
    result["image"] = img_path.name
    inat_results.append(result)

inat_correct = sum(r["correct"] for r in inat_results)
print(f"iNat overall accuracy: {inat_correct}/{len(inat_results)} "
      f"({100 * inat_correct / len(inat_results):.1f}%)")

In [ ]:
# ── Side-by-side comparison table ──
comparison_rows = []
for cls in HYMENIUM_CLASSES:
    # DB habitus results
    db_cls = [r for r in results if r["ground_truth"] == cls]
    db_n = len(db_cls)
    db_acc = sum(r["correct"] for r in db_cls) / db_n if db_n > 0 else None

    # iNat detail results
    inat_cls = [r for r in inat_results if r["ground_truth"] == cls]
    inat_n = len(inat_cls)
    inat_acc = sum(r["correct"] for r in inat_cls) / inat_n if inat_n > 0 else None

    delta = None
    if db_acc is not None and inat_acc is not None:
        delta = inat_acc - db_acc

    comparison_rows.append({
        "class": cls,
        "db_n": db_n,
        "db_acc": db_acc,
        "inat_n": inat_n,
        "inat_acc": inat_acc,
        "delta": delta,
    })

df_cmp = pd.DataFrame(comparison_rows)
df_cmp["db_acc_pct"] = df_cmp["db_acc"].apply(lambda x: f"{100*x:.0f}%" if x is not None else "N/A")
df_cmp["inat_acc_pct"] = df_cmp["inat_acc"].apply(lambda x: f"{100*x:.0f}%" if x is not None else "N/A")
df_cmp["delta_pp"] = df_cmp["delta"].apply(lambda x: f"{100*x:+.0f}pp" if x is not None else "NEW")

print("Per-class accuracy comparison:")
print(df_cmp[["class", "db_n", "db_acc_pct", "inat_n", "inat_acc_pct", "delta_pp"]].to_string(index=False))
print(f"\nOverall DB habitus:   {correct}/{len(results)} ({100*correct/len(results):.1f}%)")
print(f"Overall iNat detail:  {inat_correct}/{len(inat_results)} ({100*inat_correct/len(inat_results):.1f}%)")

In [ ]:
# ── Visual comparison ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Side-by-side bar chart
ax = axes[0]
x = np.arange(len(HYMENIUM_CLASSES))
width = 0.35

db_accs = [r["db_acc"] if r["db_acc"] is not None else 0 for r in comparison_rows]
inat_accs = [r["inat_acc"] if r["inat_acc"] is not None else 0 for r in comparison_rows]

bars1 = ax.barh(x + width / 2, db_accs, width, label="DB habitus", color="#3498db", alpha=0.8)
bars2 = ax.barh(x - width / 2, inat_accs, width, label="iNat detail", color="#e67e22", alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(HYMENIUM_CLASSES, fontsize=9)
ax.set_xlim(0, 1)
ax.set_xlabel("Accuracy")
ax.set_title("CLIP Zero-Shot: DB Habitus vs iNat Detail")
ax.axvline(0.8, color="green", linestyle="--", alpha=0.4)
ax.axvline(0.6, color="orange", linestyle="--", alpha=0.4)
ax.legend(loc="lower right", fontsize=8)

# iNat confusion matrix
ax = axes[1]
inat_confusion = pd.DataFrame(0, index=HYMENIUM_CLASSES, columns=HYMENIUM_CLASSES)
for r in inat_results:
    inat_confusion.loc[r["ground_truth"], r["predicted"]] += 1

conf_arr = inat_confusion.values.astype(float)
row_sums = conf_arr.sum(axis=1, keepdims=True)
conf_norm = np.divide(conf_arr, row_sums, where=row_sums > 0, out=np.zeros_like(conf_arr))

im = ax.imshow(conf_norm, cmap="Oranges", vmin=0, vmax=1)
ax.set_xticks(range(len(HYMENIUM_CLASSES)))
ax.set_yticks(range(len(HYMENIUM_CLASSES)))
ax.set_xticklabels(HYMENIUM_CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(HYMENIUM_CLASSES, fontsize=8)
ax.set_xlabel("Predicted")
ax.set_ylabel("Ground Truth")
ax.set_title("iNat Detail — Confusion Matrix")

for i in range(len(HYMENIUM_CLASSES)):
    for j in range(len(HYMENIUM_CLASSES)):
        val = conf_arr[i, j]
        if val > 0:
            ax.text(j, i, f"{int(val)}", ha="center", va="center",
                    color="white" if conf_norm[i, j] > 0.5 else "black", fontsize=8)

fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()